In [ ]:
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader as GNNLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from scipy.spatial import KDTree
import torch.optim as optim

# Set device globally so all classes and functions can see it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. DATASET CLASS (With SciPy & Type Fixes)
# ==========================================
class JetGraphDataset(Dataset):
    def __init__(self, file_path, num_samples=2000):
        super().__init__()
        with h5py.File(file_path, 'r') as f:
            self.images = f['X_jets'][:num_samples]
            self.labels = f['y'][:num_samples]
            
    def len(self):
        return len(self.images)

    def get(self, idx):
        img = self.images[idx] 
        label = int(self.labels[idx]) # Fixed integer casting

        coords = np.argwhere(np.sum(img, axis=-1) > 0)
        
        if len(coords) == 0:
            return Data(x=torch.zeros((1, 5)), 
                        edge_index=torch.empty((2, 0), dtype=torch.long), 
                        y=torch.tensor([label], dtype=torch.long))

        raw_hits = img[coords[:, 0], coords[:, 1]] 
        norm_coords = coords.astype(np.float32) / 125.0 
        
        x_np = np.hstack([norm_coords, raw_hits.astype(np.float32)])
        x = torch.from_numpy(x_np).float()
        
        tree = KDTree(coords)
        _, indices = tree.query(coords, k=7)
        
        row = np.repeat(np.arange(len(coords), dtype=np.int64), 6)
        col = indices[:, 1:].flatten().astype(np.int64)
        
        edge_index = torch.tensor(np.stack([row, col]), dtype=torch.long)
        return Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.long))

# ==========================================
# 2. CONTRASTIVE MODEL CLASS
# ==========================================
class ContrastiveJetGNN(nn.Module):
    def __init__(self, embedding_dim=64):
        super(ContrastiveJetGNN, self).__init__()
        self.conv1 = GCNConv(5, 32)
        self.conv2 = GCNConv(32, 64)
        self.conv3 = GCNConv(64, embedding_dim)
        
        self.projection_head = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64) 
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        representation = global_mean_pool(x, batch)
        projected = self.projection_head(representation)
        return F.normalize(projected, p=2, dim=1), representation

# ==========================================
# 3. CONTRASTIVE LOSS FUNCTION
# ==========================================
def supervised_contrastive_loss(embeddings, labels, temperature=0.07):
    similarity_matrix = torch.matmul(embeddings, embeddings.T) / temperature
    labels = labels.view(-1, 1)
    mask = torch.eq(labels, labels.T).float().to(device)
    mask = mask - torch.eye(mask.shape[0]).to(device)
    
    exp_logits = torch.exp(similarity_matrix)
    log_prob = similarity_matrix - torch.log(exp_logits.sum(1, keepdim=True))
    mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-6)
    return -mean_log_prob_pos.mean()

# ==========================================
# 4. EXECUTION & TRAINING LOOP
# ==========================================
print("Loading data...")
file_path = 'quark-gluon_data-set_n139306.hdf5'
graph_dataset = JetGraphDataset(file_path, num_samples=2000)
train_loader_gnn = GNNLoader(graph_dataset, batch_size=32, shuffle=True)

print("Initializing model...")
model_cl = ContrastiveJetGNN().to(device)
optimizer_cl = optim.Adam(model_cl.parameters(), lr=0.001)

print("Starting Contrastive Pre-training...")
for epoch in range(1, 11):
    model_cl.train()
    total_loss = 0
    for data in train_loader_gnn:
        data = data.to(device)
        optimizer_cl.zero_grad()
        projected, _ = model_cl(data)
        
        loss = supervised_contrastive_loss(projected, data.y)
        loss.backward()
        optimizer_cl.step()
        total_loss += loss.item()
        
    print(f"CL Epoch {epoch}/10 | Loss: {total_loss/len(train_loader_gnn):.4f}")
print("Training Complete!")

Loading data...
Initializing model...
🚀 Starting Contrastive Pre-training...
CL Epoch 1/10 | Loss: 3.4547
CL Epoch 2/10 | Loss: 3.4547
CL Epoch 3/10 | Loss: 3.4547
CL Epoch 4/10 | Loss: 3.4547
CL Epoch 5/10 | Loss: 3.4547
CL Epoch 6/10 | Loss: 3.4547
CL Epoch 7/10 | Loss: 3.4547
CL Epoch 8/10 | Loss: 3.4547
CL Epoch 9/10 | Loss: 3.4547
CL Epoch 10/10 | Loss: 3.4547
✅ Training Complete!


In [ ]:
# ==========================================
# 5. LINEAR PROBE (CLASSIFICATION EVALUATION)
# ==========================================
print("\nStarting Classification Evaluation...")

# Freeze the Contrastive GNN so we only train the new classifier head
for param in model_cl.parameters():
    param.requires_grad = False

classifier = nn.Linear(64, 2).to(device)
# Using a higher learning rate for the linear probe
optimizer_eval = optim.Adam(classifier.parameters(), lr=0.01) 

classifier.train()
for epoch in range(1, 6):
    correct = 0
    total = 0
    total_loss = 0
    
    for data in train_loader_gnn:
        data = data.to(device)
        optimizer_eval.zero_grad()
        
        # Get the representation from the frozen GNN
        with torch.no_grad():
            _, representation = model_cl(data)
        
        # Classify and calculate loss
        out = classifier(representation)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer_eval.step()
        
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
        total += data.y.size(0)
        
    acc = correct / total
    print(f"Eval Epoch {epoch}/5 | Loss: {total_loss/len(train_loader_gnn):.4f} | Accuracy: {acc:.4f}")

print("ALL CODING TASKS COMPLETED!")


📊 Starting Classification Evaluation...
Eval Epoch 1/5 | Loss: 0.6945 | Accuracy: 0.5020
Eval Epoch 2/5 | Loss: 0.6940 | Accuracy: 0.5040
Eval Epoch 3/5 | Loss: 0.6934 | Accuracy: 0.5010
Eval Epoch 4/5 | Loss: 0.6933 | Accuracy: 0.5005
Eval Epoch 5/5 | Loss: 0.6946 | Accuracy: 0.4930
🎉 ALL CODING TASKS COMPLETED!
